<div style="text-align:center">

# Proyecto 1  
## Clasificación de Riesgo Crediticio  

**Machine Learning**  
Universidad del Rosario

Vanessa Ochoa, Nilson Amaya, Juan Zamora

</div>


---

## Tabla de contenido

- [Introducción](#1-introducción)
- [Importación del dataset](#2-importación-del-dataset)
- [EDA](#3-análisis-exploratorio-de-datos-eda)
- [Modelos](#4-modelos)
- [Resultados](#5-resultados)
- [Conclusiones](#6-conclusiones)

<a id="1-introducción"></a>
## 1. Introducción

El riesgo crediticio es uno de los principales problemas que enfrentan las instituciones financieras, ya que implica la posibilidad de que una persona no cumpla con sus obligaciones de pago. Poder anticipar este tipo de situaciones es clave para reducir pérdidas y mejorar la toma de decisiones.

En este proyecto se utiliza el dataset **"Give Me Some Credit"** con el objetivo de predecir si una persona tendrá dificultades financieras en los próximos dos años. Este problema se aborda como una tarea de **clasificación binaria**, donde el modelo debe identificar si un cliente presenta riesgo de incumplimiento o no.

Para resolver este problema se desarrolla un flujo completo de **Machine Learning**, que incluye análisis exploratorio de datos (EDA), tratamiento de valores faltantes y outliers, implementación de modelos de clasificación como **k-NN** y **Regresión Logística**, así como la optimización de hiperparámetros mediante **Grid Search** y validación cruzada.

Finalmente, se comparan los modelos obtenidos para identificar cuál ofrece el mejor desempeño en la predicción del riesgo crediticio.

<a id="2-importación-del-dataset"></a>
## 2. Importación del dataset

Antes de cargar el dataset, se importan las librerías necesarias para realizar el proyecto.

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [17]:
# Cargamos el dataset

train = pd.read_csv("../input/cs-training.csv")
test = pd.read_csv("../input/cs-test.csv")

# ver primeras filas
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 11  NumberOfDep

### 2.1 Descripción de las variables

- **SeriousDlqin2yrs:** Indica si la persona ha tenido una morosidad grave, es decir, un atraso de 90 días o más en el pago de sus obligaciones durante los últimos dos años. Es la variable objetivo del modelo.

- **RevolvingUtilizationOfUnsecuredLines:** Proporción del saldo utilizado en tarjetas de crédito y líneas de crédito personales respecto al límite total disponible.

- **age:** Edad del solicitante del crédito expresada en años.

- **NumberOfTime30-59DaysPastDueNotWorse:** Número de veces que el prestatario ha tenido retrasos en sus pagos entre 30 y 59 días durante los últimos dos años.

- **DebtRatio:** Relación entre las obligaciones financieras mensuales (deudas, manutención, gastos) y el ingreso bruto mensual del prestatario.

- **MonthlyIncome:** Ingreso mensual reportado por el prestatario.

- **NumberOfOpenCreditLinesAndLoans:** Número total de préstamos y líneas de crédito abiertas, como tarjetas de crédito, préstamos de auto o hipotecas.

- **NumberOfTimes90DaysLate:** Número de veces que el prestatario ha tenido retrasos en pagos de 90 días o más.

- **NumberRealEstateLoansOrLines:** Número de préstamos relacionados con bienes raíces, incluyendo hipotecas y líneas de crédito sobre vivienda.

- **NumberOfTime60-89DaysPastDueNotWorse:** Número de veces que el prestatario ha tenido retrasos en pagos entre 60 y 89 días en los últimos dos años.

- **NumberOfDependents:** Número de personas dependientes económicamente del prestatario, como hijos o cónyuge.

<a id="3-análisis-exploratorio-de-datos-eda"></a>
## 3. Análisis Exploratorio de Datos (EDA)

### 3.1 Identificación y tratamiento de datos faltantes e imputación lógica

Primero identificamos el porcentaje de datos faltantes por variable y, con base en el contexto de negocio, aplicamos una imputación lógica:

- **MonthlyIncome**: se imputa con la **mediana**, porque suele tener distribución sesgada y la mediana es robusta a valores extremos.
- **NumberOfDependents**: se imputa con la **mediana** para conservar el comportamiento típico del hogar sin introducir sesgo fuerte.


In [ ]:
# Copia de trabajo para el EDA
df = train.copy()

# Eliminamos columna índice si existe
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

missing = df.isnull().sum().to_frame('nulos')
missing['porcentaje'] = (missing['nulos'] / len(df) * 100).round(2)
missing = missing[missing['nulos'] > 0].sort_values('porcentaje', ascending=False)
missing


In [ ]:
# Imputación lógica
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

print('Nulos restantes por variable:')
print(df.isnull().sum()[df.isnull().sum() > 0])


### 3.2 Análisis de distribuciones y detección de outliers

Analizamos la forma de las distribuciones con histogramas y boxplots. Para detectar valores atípicos extremos usamos el criterio del **IQR (rango intercuartílico)** y realizamos winsorización suave (capado por percentiles) en variables con colas muy largas.


In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns

# Histograma de variables numéricas
df[num_cols].hist(figsize=(16, 12), bins=30)
plt.suptitle('Distribuciones de variables numéricas', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Conteo de outliers por IQR
outlier_count = {}
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count[col] = ((df[col] < lower) | (df[col] > upper)).sum()

outlier_summary = (pd.Series(outlier_count, name='n_outliers')
                   .sort_values(ascending=False)
                   .to_frame())
outlier_summary['porcentaje'] = (outlier_summary['n_outliers'] / len(df) * 100).round(2)
outlier_summary.head(10)


In [ ]:
# Tratamiento de outliers extremos por winsorización en variables altamente sesgadas
cols_winsor = [
    'RevolvingUtilizationOfUnsecuredLines',
    'DebtRatio',
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate'
]

for col in cols_winsor:
    if col in df.columns:
        p01, p99 = df[col].quantile([0.01, 0.99])
        df[col] = df[col].clip(lower=p01, upper=p99)

plt.figure(figsize=(14, 6))
sns.boxplot(data=df[cols_winsor])
plt.xticks(rotation=45, ha='right')
plt.title('Boxplot después de winsorización (1%-99%)')
plt.tight_layout()
plt.show()


### 3.3 Análisis de correlación y selección de características justificadas

Calculamos la correlación de Pearson entre variables numéricas y observamos su relación con la variable objetivo (**SeriousDlqin2yrs**). La selección inicial de características se basa en:

1. **Relevancia estadística** (magnitud de correlación con el objetivo).
2. **No redundancia** (evitar pares altamente correlacionados entre sí).
3. **Interpretabilidad de negocio** (variables de mora, utilización de crédito e ingresos son explicativas del riesgo).


In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

corr_target = corr['SeriousDlqin2yrs'].drop('SeriousDlqin2yrs').sort_values(key=lambda x: x.abs(), ascending=False)
corr_target.head(10)


In [ ]:
# Selección justificada de características
selected_features = [
    'RevolvingUtilizationOfUnsecuredLines',
    'age',
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
    'DebtRatio',
    'MonthlyIncome',
    'NumberOfOpenCreditLinesAndLoans',
    'NumberRealEstateLoansOrLines',
    'NumberOfDependents'
]

X = df[selected_features]
y = df['SeriousDlqin2yrs']

print('Variables seleccionadas para modelado:')
print(selected_features)
print('\nDimensiones de X e y:', X.shape, y.shape)


<a id="4-modelos"></a>
## 4. Implementación de Modelos

<a id="5-resultados"></a>
## 5. Optimización de Hiperparámetros y Validación

<a id="6-conclusiones"></a>
## 6. Métricas de Evaluación y Decisión